# Evaluate Model Performance

This notebook demonstrates how to evaluate trained models and visualize results with comprehensive metrics.

## Setup and Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

# Import project modules
from src.config import get_config
from src.data_processing import generate_synthetic_egm_data, generate_synthetic_labels
from src.utils import ensure_output_directory

import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

## 1. Load Configuration

In [ ]:
config = get_config()
config.load_from_yaml('config.yaml')

print("Configuration loaded.")

## 2. Generate Mock Predictions

For demonstration, we'll use synthetic predictions. In practice, use actual model predictions.

In [ ]:
# Generate test data
print("Generating test data...")
n_test_samples = 100
n_classes = 3

# True labels
y_true = generate_synthetic_labels(
    n_test_samples, n_classes=n_classes, class_balance=0.30, random_state=42
)

# Simulate model predictions (probabilistic)
# In practice: y_pred = model.predict(X_test)
np.random.seed(42)
y_pred = np.random.rand(n_test_samples, n_classes)

# Make predictions more realistic (better for samples with true labels)
for i in range(n_test_samples):
    for j in range(n_classes):
        if y_true[i, j] == 1:
            y_pred[i, j] += np.random.uniform(0.2, 0.4)  # Boost positive predictions
        else:
            y_pred[i, j] -= np.random.uniform(0.1, 0.2)  # Reduce negative predictions

# Clip to [0, 1]
y_pred = np.clip(y_pred, 0, 1)

print(f"✓ Generated test data:")
print(f"  True labels shape: {y_true.shape}")
print(f"  Predictions shape: {y_pred.shape}")

## 3. Evaluation Metrics Functions

Define comprehensive evaluation metrics and compute class-specific thresholds using Youden's J statistic.

In [ ]:
def find_optimal_thresholds_youden_j(y_true, y_pred, threshold_grid=None):
    """
    Select per-class probability thresholds by maximizing Youden's J statistic.
    J = sensitivity - (1 - specificity) = TPR - FPR
    """
    if threshold_grid is None:
        threshold_grid = np.linspace(0.0, 1.0, 201)
    
    n_classes = y_true.shape[1]
    best_thresholds = np.zeros(n_classes, dtype=float)
    youden_details = {}
    
    for class_idx in range(n_classes):
        y_t = y_true[:, class_idx]
        y_p = y_pred[:, class_idx]
        
        tprs, fprs, j_values = [], [], []
        
        for thresh in threshold_grid:
            y_p_binary = (y_p >= thresh).astype(int)
            tp = np.sum((y_p_binary == 1) & (y_t == 1))
            tn = np.sum((y_p_binary == 0) & (y_t == 0))
            fp = np.sum((y_p_binary == 1) & (y_t == 0))
            fn = np.sum((y_p_binary == 0) & (y_t == 1))
            
            tpr = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            j = tpr - fpr
            
            tprs.append(tpr)
            fprs.append(fpr)
            j_values.append(j)
        
        best_idx = int(np.argmax(j_values))
        best_thresholds[class_idx] = float(threshold_grid[best_idx])
        
        youden_details[class_idx] = {
            'thresholds': np.array(threshold_grid),
            'tprs': np.array(tprs),
            'fprs': np.array(fprs),
            'j_values': np.array(j_values),
            'best_index': best_idx,
            'best_threshold': float(threshold_grid[best_idx]),
            'best_tpr': float(tprs[best_idx]),
            'best_fpr': float(fprs[best_idx]),
            'best_j': float(j_values[best_idx]),
        }
    
    return best_thresholds, youden_details


def compute_metrics(y_true, y_pred, threshold=0.5, per_class_thresholds=None):
    """
    Compute comprehensive evaluation metrics.
    
    Parameters:
    -----------
    y_true : (n_samples, n_classes) array
        Ground truth binary labels
    y_pred : (n_samples, n_classes) array
        Predicted probabilities (0-1)
    threshold : float
        Global classification threshold used when per_class_thresholds is None
    per_class_thresholds : array-like or None
        Optional class-specific thresholds
    
    Returns:
    --------
    metrics : dict
        Comprehensive metrics dictionary
    """
    n_classes = y_true.shape[1]
    
    if per_class_thresholds is None:
        thresholds = np.full(n_classes, float(threshold))
    else:
        thresholds = np.array(per_class_thresholds, dtype=float)
        if thresholds.shape[0] != n_classes:
            raise ValueError('per_class_thresholds must match the number of classes')
    
    metrics = {'per_class': {}}
    
    # Per-class metrics
    for class_idx in range(n_classes):
        y_t = y_true[:, class_idx]
        y_p_prob = y_pred[:, class_idx]
        class_threshold = thresholds[class_idx]
        y_p = (y_p_prob >= class_threshold).astype(int)
        
        tp = np.sum((y_p == 1) & (y_t == 1))
        tn = np.sum((y_p == 0) & (y_t == 0))
        fp = np.sum((y_p == 1) & (y_t == 0))
        fn = np.sum((y_p == 0) & (y_t == 1))
        
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = 2 * (precision * sensitivity) / (precision + sensitivity) if (precision + sensitivity) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)
        
        # Area under ROC curve (approximate)
        auc_estimate = np.where(y_t == 1, y_p_prob.mean(), 1 - y_p_prob.mean()).mean()
        
        metrics['per_class'][class_idx] = {
            'threshold': float(class_threshold),
            'sensitivity': sensitivity,
            'specificity': specificity,
            'precision': precision,
            'f1_score': f1,
            'accuracy': accuracy,
            'auc_estimate': auc_estimate,
            'tp': int(tp),
            'tn': int(tn),
            'fp': int(fp),
            'fn': int(fn)
        }
    
    # Macro-averaged metrics (average across classes)
    metrics['macro'] = {
        'threshold': float(np.mean(thresholds)),
        'sensitivity': np.mean([m['sensitivity'] for m in metrics['per_class'].values()]),
        'specificity': np.mean([m['specificity'] for m in metrics['per_class'].values()]),
        'precision': np.mean([m['precision'] for m in metrics['per_class'].values()]),
        'f1_score': np.mean([m['f1_score'] for m in metrics['per_class'].values()]),
        'accuracy': np.mean([m['accuracy'] for m in metrics['per_class'].values()]),
        'auc_estimate': np.mean([m['auc_estimate'] for m in metrics['per_class'].values()])
    }
    
    return metrics


optimal_thresholds, youden_info = find_optimal_thresholds_youden_j(y_true, y_pred)
metrics = compute_metrics(y_true, y_pred, per_class_thresholds=optimal_thresholds)

print("✓ Metrics computed using Youden's J optimal thresholds")
for class_idx, thresh in enumerate(optimal_thresholds):
    print(f"  Class {class_idx}: threshold={thresh:.3f}, J={youden_info[class_idx]['best_j']:.3f}")

## 4. Display Per-Class Metrics

In [ ]:
# Create metrics table
class_names = ['Epicardium', 'Myocardium', 'Endocardium']
rows = []

for class_idx, class_name in enumerate(class_names[:len(metrics['per_class'])]):
    m = metrics['per_class'][class_idx]
    rows.append({
        'Class': class_name,
        'Threshold (Youden J)': f"{m['threshold']:.3f}",
        'Sensitivity': f"{m['sensitivity']:.3f}",
        'Specificity': f"{m['specificity']:.3f}",
        'Precision': f"{m['precision']:.3f}",
        'F1-Score': f"{m['f1_score']:.3f}",
        'Accuracy': f"{m['accuracy']:.3f}",
        'AUC': f"{m['auc_estimate']:.3f}"
    })

df_metrics = pd.DataFrame(rows)

# Add macro average
macro = metrics['macro']
df_metrics.loc[len(df_metrics)] = {
    'Class': 'MACRO AVG',
    'Threshold (Youden J)': f"{macro['threshold']:.3f}",
    'Sensitivity': f"{macro['sensitivity']:.3f}",
    'Specificity': f"{macro['specificity']:.3f}",
    'Precision': f"{macro['precision']:.3f}",
    'F1-Score': f"{macro['f1_score']:.3f}",
    'Accuracy': f"{macro['accuracy']:.3f}",
    'AUC': f"{macro['auc_estimate']:.3f}"
}

print("\nPer-Class Performance Metrics (Youden's J thresholds):")
print("=" * 100)
print(df_metrics.to_string(index=False))
print("=" * 100)

## 5. Confusion Matrix Visualization (Youden's J Thresholds)

In [ ]:
# Binarize predictions using class-specific Youden thresholds
y_pred_binary = (y_pred >= optimal_thresholds.reshape(1, -1)).astype(int)

# Plot confusion matrices for each class
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_names = ['Epicardium', 'Myocardium', 'Endocardium']

for class_idx, ax in enumerate(axes):
    y_t = y_true[:, class_idx]
    y_p = y_pred_binary[:, class_idx]
    
    # Compute confusion matrix
    cm = np.zeros((2, 2))
    cm[0, 0] = np.sum((y_p == 0) & (y_t == 0))  # TN
    cm[0, 1] = np.sum((y_p == 1) & (y_t == 0))  # FP
    cm[1, 0] = np.sum((y_p == 0) & (y_t == 1))  # FN
    cm[1, 1] = np.sum((y_p == 1) & (y_t == 1))  # TP
    
    # Plot
    sns.heatmap(
        cm,
        annot=True,
        fmt='.0f',
        cmap='Blues',
        ax=ax,
        xticklabels=['Negative', 'Positive'],
        yticklabels=['Negative', 'Positive'],
        cbar=False
    )
    ax.set_title(f"{class_names[class_idx]}\nthreshold={optimal_thresholds[class_idx]:.3f}")
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()
print("\nConfusion matrices displayed using Youden's J thresholds for each cardiac layer.")

## 6. ROC Curve Analysis

In [ ]:
# Plot ROC curves and mark Youden's J operating points
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
class_names = ['Epicardium', 'Myocardium', 'Endocardium']

for class_idx, ax in enumerate(axes):
    details = youden_info[class_idx]
    fprs = details['fprs']
    tprs = details['tprs']
    best_idx = details['best_index']
    best_fpr = details['best_fpr']
    best_tpr = details['best_tpr']
    best_threshold = details['best_threshold']
    best_j = details['best_j']
    
    ax.plot(fprs, tprs, linewidth=2, label='Model ROC')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
    ax.scatter([best_fpr], [best_tpr], color='red', s=50, zorder=5, label="Youden's J optimum")
    ax.annotate(
        f"thr={best_threshold:.2f}\nJ={best_j:.2f}",
        xy=(best_fpr, best_tpr),
        xytext=(8, -8),
        textcoords='offset points',
        fontsize=9,
        color='red'
    )
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC - {class_names[class_idx]}')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("\nROC curves displayed with Youden's J optimal operating points.")